# Advanced Agentic Patterns: Routing, Orchestration & MCP

## Beyond Simple Loops

Most agentic tutorials show a basic loop: **LLM → Tool → LLM → Tool**. Production systems require sophisticated **routing**, **multi-agent orchestration**, and **standardized tool protocols**. This module teaches enterprise-grade patterns.

## Part 1: Routing Agents (Smart Tool Selection)

### The Problem: Which Tool to Use?



### The Solution: Semantic Routing

Route queries to specialized agents based on intent:



**Benefits:**
- 10x faster (no tool enumeration)
- Cheaper (fewer LLM calls)
- More reliable (specialized agents)

---

In [ ]:
# Naive approach: LLM decides every time
def naive_agent(query):
    response = llm.invoke(f"Use tools to answer: {query}")
    # Slow, expensive, unreliable
    return response

In [ ]:
from enum import Enum
import json

class AgentType(Enum):
    SEARCH = "search"
    CALCULATOR = "calculator"
    DATABASE = "database"
    SUMMARIZER = "summarizer"

def route_query(query: str) -> AgentType:
    """Route query to appropriate agent"""
    routing_prompt = f"""
    Classify this query into ONE category:
    - SEARCH: Information retrieval, web queries
    - CALCULATOR: Math, computations
    - DATABASE: Data lookups, SQL queries
    - SUMMARIZER: Text summarization, analysis
    
    Query: {query}
    
    Respond with ONLY the category name.
    """
    
    response = llm.invoke(routing_prompt)
    category = response.strip().upper()
    return AgentType[category]

# Specialized agents
class SearchAgent:
    def run(self, query):
        results = search_api.query(query)
        return f"Found {len(results)} results: {results[:3]}"

class CalculatorAgent:
    def run(self, query):
        # Extract math expression and compute
        expr = extract_math(query)
        return f"Result: {eval(expr)}"

class DatabaseAgent:
    def run(self, query):
        sql = llm.invoke(f"Convert to SQL: {query}")
        return db.execute(sql)

# Router
agents = {
    AgentType.SEARCH: SearchAgent(),
    AgentType.CALCULATOR: CalculatorAgent(),
    AgentType.DATABASE: DatabaseAgent(),
}

def intelligent_agent(query):
    agent_type = route_query(query)
    agent = agents[agent_type]
    return agent.run(query)

# Test
print(intelligent_agent("What is 2 + 2?"))  # → CalculatorAgent
print(intelligent_agent("Find AI news"))     # → SearchAgent

## Part 2: Multi-Agent Orchestration (Manager/Worker Pattern)

### Architecture: Manager Coordinates Workers



**Output:**
```
DataAnalyst: Analyzed throughput requirements (1000 req/s)
Engineer: Proposed vLLM + Kubernetes architecture
Researcher: Cited 3 recent papers on distributed inference
Synthesis: Recommended vLLM on K8s with auto-scaling...
```

---

In [ ]:
from typing import List
import asyncio

class WorkerAgent:
    def __init__(self, name: str, expertise: str):
        self.name = name
        self.expertise = expertise
    
    async def execute(self, task: str) -> str:
        """Execute task within expertise"""
        prompt = f"""
        You are {self.name}, expert in {self.expertise}.
        Task: {task}
        Provide a focused, expert response.
        """
        return await llm.ainvoke(prompt)

class ManagerAgent:
    def __init__(self, workers: List[WorkerAgent]):
        self.workers = workers
    
    async def decompose(self, goal: str) -> List[str]:
        """Break goal into subtasks"""
        prompt = f"""
        Break this goal into 3-5 independent subtasks:
        Goal: {goal}
        
        Format as JSON:
        {{"subtasks": ["task1", "task2", ...]}}
        """
        response = await llm.ainvoke(prompt)
        return json.loads(response)["subtasks"]
    
    async def orchestrate(self, goal: str) -> str:
        """Coordinate workers to achieve goal"""
        # Step 1: Decompose
        subtasks = await self.decompose(goal)
        
        # Step 2: Assign to workers
        tasks = []
        for i, subtask in enumerate(subtasks):
            worker = self.workers[i % len(self.workers)]
            tasks.append(worker.execute(subtask))
        
        # Step 3: Execute in parallel
        results = await asyncio.gather(*tasks)
        
        # Step 4: Synthesize
        synthesis_prompt = f"""
        Synthesize these results into a coherent answer:
        {json.dumps(dict(zip(subtasks, results)))}
        """
        return await llm.ainvoke(synthesis_prompt)

# Usage
workers = [
    WorkerAgent("DataAnalyst", "data analysis and statistics"),
    WorkerAgent("Engineer", "system design and architecture"),
    WorkerAgent("Researcher", "literature review and trends"),
]

manager = ManagerAgent(workers)

# Run
result = asyncio.run(manager.orchestrate(
    "Design a scalable ML inference system"
))
print(result)

## Part 3: Model Context Protocol (MCP) for Standardized Tools

### The Problem: Tool Integration is Messy



### The Solution: MCP Standard

MCP defines a **universal tool interface**:



**Benefits:**
- **Standardized:** All tools follow same interface
- **Composable:** Mix and match tools easily
- **Debuggable:** Clear input/output contracts
- **Scalable:** Add tools without changing agent code

---

In [ ]:
# Without MCP: Each tool has different interface
def use_calculator(expr):
    return eval(expr)

def use_search(query):
    return search_api.query(query)

def use_database(sql):
    return db.execute(sql)

# LLM must learn each interface → fragile

In [ ]:
from typing import Any, Dict
import json

class MCPTool:
    """Model Context Protocol Tool"""
    def __init__(self, name: str, description: str, schema: Dict):
        self.name = name
        self.description = description
        self.schema = schema  # JSON Schema for inputs
    
    def invoke(self, params: Dict) -> str:
        """Execute tool with validated params"""
        raise NotImplementedError

class CalculatorTool(MCPTool):
    def __init__(self):
        super().__init__(
            name="calculator",
            description="Perform mathematical calculations",
            schema={
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression (e.g., '2 + 2')"
                    }
                },
                "required": ["expression"]
            }
        )
    
    def invoke(self, params: Dict) -> str:
        try:
            result = eval(params["expression"])
            return json.dumps({"result": result, "error": None})
        except Exception as e:
            return json.dumps({"result": None, "error": str(e)})

class SearchTool(MCPTool):
    def __init__(self):
        super().__init__(
            name="search",
            description="Search the web for information",
            schema={
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query"
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Max results (default 5)"
                    }
                },
                "required": ["query"]
            }
        )
    
    def invoke(self, params: Dict) -> str:
        results = search_api.query(params["query"], limit=params.get("limit", 5))
        return json.dumps({"results": results, "error": None})

# Tool registry
tools = {
    "calculator": CalculatorTool(),
    "search": SearchTool(),
}

# LLM uses standardized interface
def mcp_agent(query: str) -> str:
    """Agent using MCP tools"""
    messages = [{"role": "user", "content": query}]
    
    # Tell LLM about available tools
    tools_description = json.dumps([
        {
            "name": tool.name,
            "description": tool.description,
            "input_schema": tool.schema
        }
        for tool in tools.values()
    ])
    
    system_prompt = f"""
    You have access to these tools:
    {tools_description}
    
    When you need to use a tool, respond with:
    <tool_use>
    {{"name": "tool_name", "params": {{...}}}}
    </tool_use>
    """
    
    response = llm.invoke(system_prompt + query)
    
    # Parse tool calls
    if "<tool_use>" in response:
        tool_call = json.loads(response.split("<tool_use>")[1].split("</tool_use>")[0])
        tool = tools[tool_call["name"]]
        result = tool.invoke(tool_call["params"])
        return f"Tool result: {result}"
    
    return response

# Test
print(mcp_agent("What is 2 + 2?"))
print(mcp_agent("Search for AI news"))

## Part 4: Production Patterns

### Pattern 1: Retry with Exponential Backoff



### Pattern 2: Timeout Protection



### Pattern 3: Cost Tracking



---

In [ ]:
import asyncio
from functools import wraps

def retry_with_backoff(max_retries=3, base_delay=1):
    def decorator(func):
        @wraps(func)
        async def wrapper(*args, **kwargs):
            for attempt in range(max_retries):
                try:
                    return await func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_retries - 1:
                        raise
                    delay = base_delay * (2 ** attempt)
                    await asyncio.sleep(delay)
        return wrapper
    return decorator

@retry_with_backoff(max_retries=3)
async def call_llm_with_retry(prompt):
    return await llm.ainvoke(prompt)

In [ ]:
async def agent_with_timeout(query: str, timeout_sec: int = 30) -> str:
    try:
        return await asyncio.wait_for(
            orchestrate(query),
            timeout=timeout_sec
        )
    except asyncio.TimeoutError:
        return "Agent timed out. Please try a simpler query."

In [ ]:
class CostTracker:
    def __init__(self):
        self.total_cost = 0.0
        self.calls = {}
    
    def track_call(self, model: str, input_tokens: int, output_tokens: int):
        # Pricing: Claude 3 Sonnet
        input_cost = input_tokens * 0.003 / 1000
        output_cost = output_tokens * 0.015 / 1000
        cost = input_cost + output_cost
        
        self.total_cost += cost
        self.calls[model] = self.calls.get(model, 0) + cost
        
        print(f"Call cost: ${cost:.4f} | Total: ${self.total_cost:.2f}")

tracker = CostTracker()
tracker.track_call("claude-3-sonnet", 500, 200)

## Key Concepts

| Pattern | Use Case | Complexity |
|---------|----------|-----------|
| **Routing** | Intent-based tool selection | Low |
| **Manager/Worker** | Complex multi-step tasks | Medium |
| **MCP** | Standardized tool interface | Medium |
| **Retry/Timeout** | Production reliability | Low |
| **Cost Tracking** | Budget management | Low |

---

## Quizzes

### Quiz 1: Routing Benefits
**Question:** Why is semantic routing faster than asking LLM to choose tools?
- A) Fewer LLM calls + specialized agents ✓
- B) Routing is always faster
- C) LLM is slow at decision-making
- D) Routing uses GPU acceleration

### Quiz 2: Manager/Worker Pattern
**Question:** When should you use Manager/Worker orchestration?
- A) Complex goals requiring multiple specialized agents ✓
- B) Simple single-tool tasks
- C) Always, for consistency
- D) Never, it's too complex

### Quiz 3: MCP Standard
**Question:** What is the main benefit of MCP?
- A) Standardized tool interface reduces coupling ✓
- B) Faster execution
- C) Lower costs
- D) Better accuracy

---

## Resources & References

- **[Model Context Protocol](https://modelcontextprotocol.io/)** - Official MCP spec
- **[LangChain Agents](https://python.langchain.com/docs/modules/agents/)** - Agent frameworks
- **[Claude Tool Use](https://docs.anthropic.com/claude/reference/tool-use)** - Tool integration
- **[Papers with Code: Multi-Agent Systems](https://paperswithcode.com/)** - Research implementations